# Generate statistics

## Run this for both HiRES or CHARM experiments

In [ ]:
library(tidyverse)

config_path <- "provenance/effective_config.json"
if (!file.exists(config_path)) {
  stop("missing current workflow config: ", config_path)
}
config <- jsonlite::read_json(config_path, simplifyVector = TRUE)
experiment_type <- tolower(config$experiment_type)
if (!(experiment_type %in% c("charm", "hires"))) {
  stop("experiment_type must be charm or hires")
}

to_gigabases <- function(raw_bp) {
  raw_bp / 4 * 300 / 1e9
}

extract_sample <- function(path, slice_position, strip_after = NULL) {
  sample <- str_split(path, "/", simplify = TRUE)[, slice_position]
  if (!is.null(strip_after)) {
    sample <- str_split(sample, strip_after, simplify = TRUE)[, 1]
  }
  sample
}

read_read_stat <- function(path, slice_position = 3) {
  read_table2(path, col_names = FALSE) %>%
    arrange(X1) %>%
    mutate(
      sample_id = extract_sample(X1, slice_position),
      gigabases = to_gigabases(X2)
    ) %>%
    select(sample_id, gigabases)
}

read_pair_stat <- function(path, slice_position = 3, suffix_to_remove = NULL) {
  read_table2(path, col_names = FALSE) %>%
    arrange(X1) %>%
    mutate(
      sample_id = extract_sample(X1, slice_position),
      sample_id = if (!is.null(suffix_to_remove)) str_remove(sample_id, fixed(suffix_to_remove)) else sample_id
    ) %>%
    select(sample_id, pairs = X2)
}

raw_reads <- read_read_stat("stat/raw.fq.stat") %>%
  rename(Rawreads = gigabases)
dna_reads <- read_read_stat("stat/dna.fq.stat") %>%
  rename(DNAreads = gigabases)
rna_reads <- read_read_stat("stat/rna.fq.stat") %>%
  rename(RNAreads = gigabases)

raw_pairs <- read_pair_stat("stat/raw.pairs.stat") %>%
  rename(raw_pairs = pairs)
pairs_dedup <- read_pair_stat("stat/pairs.dedup.stat") %>%
  rename(pairs_dedup = pairs)

pairs_clean1 <- read_pair_stat("stat/pairs.c1.stat", slice_position = 5, suffix_to_remove = ".pairs.gz") %>%
  rename(pairs_clean1 = pairs)
pairs_clean2 <- read_pair_stat("stat/pairs.c12.stat", slice_position = 5, suffix_to_remove = ".pairs.gz") %>%
  rename(pairs_clean2 = pairs)
pairs_clean3 <- read_pair_stat("stat/pairs.c123.stat", slice_position = 5, suffix_to_remove = ".pairs.gz") %>%
  rename(pairs_clean3 = pairs)
inter_pairs_clean3 <- read_pair_stat("stat/inter.pairs.c123.stat", slice_position = 5, suffix_to_remove = ".pairs.gz") %>%
  rename(inter_pairs_clean3 = pairs)
yperx <- read_table2("stat/yperx.stat", col_names = FALSE) %>%
  arrange(X1) %>%
  mutate(sample_id = extract_sample(X1, 2)) %>%
  select(sample_id, yperx = X2)

stat <- raw_reads %>%
  left_join(dna_reads, by = "sample_id") %>%
  left_join(rna_reads, by = "sample_id") %>%
  left_join(yperx, by = "sample_id") %>%
  left_join(raw_pairs, by = "sample_id") %>%
  left_join(pairs_dedup, by = "sample_id") %>%
  left_join(pairs_clean1, by = "sample_id") %>%
  left_join(pairs_clean2, by = "sample_id") %>%
  left_join(pairs_clean3, by = "sample_id") %>%
  left_join(inter_pairs_clean3, by = "sample_id")

rna_output_types <- unlist(config$rna_output_types)
if (length(rna_output_types) == 0 || anyDuplicated(rna_output_types)) {
  stop("rna_output_types must be a non-empty unique list")
}
primary_rna_output_type <- config$rna_primary_output_type
if (length(primary_rna_output_type) != 1 || !(primary_rna_output_type %in% rna_output_types)) {
  stop("rna_primary_output_type must be one of the selected rna_output_types")
}

rna_gene_counts <- read_table2(paste0("../result/RNA_Res/", primary_rna_output_type,
                                     "/counts.gene.total.format.tsv"))
rna_gene_matrix <- as.data.frame(rna_gene_counts %>% select(-gene))
feature_stat_gene <- tibble(
  sample_id = names(rna_gene_matrix),
  RNA_primary_output_type = primary_rna_output_type,
  UMIs_gene = colSums(rna_gene_matrix),
  genes_gene = colSums(rna_gene_matrix != 0)
)

# RNA per-cell stats: annotation rate and dedup rate
rna_reads_path <- "stat/rna.reads_per_cell.stat"
if (file.exists(rna_reads_path)) {
  rna_per_cell <- read_tsv(rna_reads_path, col_names = c("sample_id", "rna_total_mapped_reads", "rna_assigned_reads"),
                           col_types = "cdd")
  feature_stat_gene <- feature_stat_gene %>%
    left_join(rna_per_cell, by = "sample_id") %>%
    mutate(
      rna_annotation_rate = ifelse(rna_total_mapped_reads > 0,
                                   rna_assigned_reads / rna_total_mapped_reads * 100,
                                   NA_real_),
      rna_dedup_rate = ifelse(rna_assigned_reads > 0,
                              (1 - UMIs_gene / rna_assigned_reads) * 100,
                              NA_real_)
    )
}

# RNA DNA contamination rate (GATC pattern in clean R2 reads)
rna_contam_path <- "stat/rna.dna_contam.stat"
if (file.exists(rna_contam_path)) {
  rna_contam <- read_tsv(rna_contam_path, col_names = c("sample_id", "rna_clean_reads", "rna_gatc_reads"),
                         col_types = "cdd")
  feature_stat_gene <- feature_stat_gene %>%
    left_join(rna_contam, by = "sample_id") %>%
    mutate(
      rna_dna_contam_rate = ifelse(rna_clean_reads > 0,
                                   rna_gatc_reads / rna_clean_reads * 100,
                                   NA_real_)
    )
}

rna_exon_counts <- read_table2(paste0("../result/RNA_Res/", primary_rna_output_type,
                                     "/counts.exon.total.format.tsv"))
rna_exon_matrix <- as.data.frame(rna_exon_counts %>% select(-gene))
feature_stat_exon <- tibble(
  sample_id = names(rna_exon_matrix),
  UMIs_exon = colSums(rna_exon_matrix),
  genes_exon = colSums(rna_exon_matrix != 0)
)

stat <- stat %>%
  left_join(feature_stat_gene, by = "sample_id") %>%
  left_join(feature_stat_exon, by = "sample_id")

stat <- stat %>%
  rename(cellname = sample_id)

if (config$if_structure) {
  rmsd <- read_lines("stat/rmsd.info") %>%
    tibble(line = .) %>%
    mutate(
      value = as.numeric(str_extract(line, "[-+]?[0-9]*\\.?[0-9]+(?:[eE][-+]?[0-9]+)?$")),
      path = str_extract(line, ".*(?=:\\s*\\[M::__main__\\])"),
      path = if_else(is.na(path), str_extract(line, "^[^\\s]+"), path)
    ) %>%
    filter(!is.na(path), !is.na(value)) %>%
    mutate(
      m = str_match(path, ".*/3d_info/([^/]+)/[^/]*\\.([^.]+)\\.align\\.rms\\.info$"),
      cellname = m[, 2],
      resolution = m[, 3]
    ) %>%
    filter(!is.na(cellname), !is.na(resolution)) %>%
    select(cellname, resolution, rmsd = value) %>%
    distinct() %>%
    pivot_wider(names_from = resolution, values_from = rmsd, names_prefix = "rmsd_") %>%
    arrange(cellname)

  stat <- stat %>% left_join(rmsd, by = "cellname")
}

if (experiment_type == "charm") {
  charm_reads <- map(
    names(config$split),
    function(split_name) {
      read_csv(paste0("stat/", split_name, ".read.stat"), col_names = FALSE) %>%
        transmute(
          cellname = X1,
          !!paste0(split_name, "_reads") := X2 / 2 * 300 / 1e9
        )
    }
  ) %>%
    reduce(full_join, by = "cellname")

  stat <- stat %>% full_join(charm_reads, by = "cellname")

  # ATAC/CUT&Tag per-cell dedup rates
  charm_dedup <- map(
    names(config$split),
    function(split_name) {
      path <- paste0("stat/", split_name, ".dedup_rate.stat")
      if (file.exists(path)) {
        read_tsv(path, col_names = c("cellname", paste0(split_name, "_dedup_rate")),
                 col_types = "cd")
      } else {
        tibble(cellname = character())
      }
    }
  ) %>% reduce(full_join, by = "cellname")

  stat <- stat %>% full_join(charm_dedup, by = "cellname")
}

expected_cells <- read_tsv("input_contract/discovered_cells.tsv", show_col_types = FALSE) %>%
  pull(sample_name)
if (nrow(stat) != length(expected_cells) || anyDuplicated(stat$cellname) ||
    !setequal(stat$cellname, expected_cells)) {
  stop("metadata cell set does not match the frozen input contract")
}
stat <- stat %>% dplyr::slice(match(expected_cells, cellname))


In [ ]:
fill_numeric <- function(x) {
  x[is.na(x) | is.nan(x)] <- 0
  x
}

qc_metrics <- stat %>%
  mutate(
    experiment_type = experiment_type,
    RNAreadsRatio = RNAreads / (RNAreads + DNAreads),
    pairsPerRead = raw_pairs / DNAreads / 1e9 * 300,
    pairsValidRatio = ifelse(raw_pairs > 0, pairs_clean3 / raw_pairs, 0),
    interPairsRatio = ifelse(pairs_clean3 > 0, inter_pairs_clean3 / pairs_clean3, 0)
  ) %>%
  select(cellname, experiment_type, everything())

qc_metrics <- qc_metrics %>%
  mutate(across(where(is.numeric), fill_numeric))


In [ ]:
qc_metrics


## Run below if this is a CHARM experiment

In [ ]:
if (experiment_type == "charm") {
suppressPackageStartupMessages({
  library(Signac)
  library(Seurat)
  library(EnsDb.Mmusculus.v79)
  library(BSgenome.Mmusculus.UCSC.mm10)
  library(future)
})

plan("multicore", workers = 10)
}


In [ ]:
if (experiment_type == "charm") {
charm <- rna_gene_counts %>%
  column_to_rownames("gene") %>%
  as.matrix() %>%
  CreateSeuratObject(assay = "rna", min.cells = 0, min.features = 0)
}


In [ ]:
if (experiment_type == "charm") {
cell_names <- intersect(
  rownames(charm@meta.data),
  colnames(rna_gene_counts %>% dplyr::select(-gene))
)
}


In [ ]:
if (experiment_type == "charm") {
atac_fragments <- CreateFragmentObject("../result/fragments/atac.fragments.bgz", cells = cell_names)
ct_fragments <- CreateFragmentObject("../result/fragments/ct.fragments.bgz", cells = cell_names)
mm10_genome <- seqlengths(BSgenome.Mmusculus.UCSC.mm10)
mm10_seqinfo <- GenomeInfoDb::Seqinfo(seqnames = names(mm10_genome), seqlengths = as.integer(mm10_genome))
atac_count_matrix <- GenomeBinMatrix(atac_fragments, binsize = 5000, genome = mm10_genome)
ct_count_matrix <- GenomeBinMatrix(ct_fragments, binsize = 5000, genome = mm10_genome)

atac_assay <- CreateChromatinAssay(counts = atac_count_matrix, fragments = atac_fragments, genome = mm10_seqinfo)
ct_assay <- CreateChromatinAssay(counts = ct_count_matrix, fragments = ct_fragments, genome = mm10_seqinfo)

charm[["atac"]] <- atac_assay
charm[["ct"]] <- ct_assay
}


In [ ]:
if (experiment_type == "charm") {
charm@meta.data <- charm@meta.data %>%
  rownames_to_column("cellname")
rownames(charm@meta.data) <- charm@meta.data$cellname
}


In [ ]:
if (experiment_type == "charm") {
options(future.globals.maxSize = 10 * 1024^3)
# calc tss enrichment
annotations <- GetGRangesFromEnsDb(ensdb = EnsDb.Mmusculus.v79)
canonical_ensembl_levels <- c(as.character(1:19), "X", "Y", "MT")
annotations <- GenomeInfoDb::keepSeqlevels(
  annotations, intersect(canonical_ensembl_levels, seqlevels(annotations)),
  pruning.mode = "coarse"
)
ensembl_levels <- seqlevels(annotations)
ucsc_levels <- ifelse(ensembl_levels == "MT", "chrM", paste0("chr", ensembl_levels))
names(ucsc_levels) <- ensembl_levels
annotations <- GenomeInfoDb::renameSeqlevels(annotations, ucsc_levels)
seqinfo(annotations) <- mm10_seqinfo[seqlevels(annotations)]

Annotation(charm[["atac"]]) <- annotations
Annotation(charm[["ct"]]) <- annotations

charm <- TSSEnrichment(charm, assay = "atac", fast = FALSE)
charm@meta.data$TSS.enrichment.atac <- charm@meta.data$TSS.enrichment

charm <- TSSEnrichment(charm, assay = "ct", fast = FALSE)
charm@meta.data$TSS.enrichment.ct <- charm@meta.data$TSS.enrichment
charm@meta.data$TSS.enrichment <- NULL

qc_metrics <- qc_metrics %>% full_join(
  charm@meta.data %>% dplyr::select(cellname, nCount_atac, nCount_ct, TSS.enrichment.atac, TSS.enrichment.ct),
  by = "cellname"
)
}


In [ ]:
options(repr.matrix.max.cols = 100)
qc_metrics


In [ ]:
if (nrow(qc_metrics) != length(expected_cells) || anyDuplicated(qc_metrics$cellname) ||
    !setequal(qc_metrics$cellname, expected_cells)) {
  stop("final metadata cell set does not match the frozen input contract")
}
qc_metrics <- qc_metrics %>% dplyr::slice(match(expected_cells, cellname))
qc_metrics %>% write_tsv("metadata_raw.tsv")
